# Task 3 — Integration and cross-domain analysis

Run cells from top to bottom in a fresh Python/Colab session. Dependencies: pandas >= 2, numpy, requests and plotly >= 5.5.

**Inputs:** `task1_building_permits_backup.zip` and `task2_street_summaries.zip` or `task2_corrected_results.zip`.

Saved raw snapshots are the reproducible inputs. Live API extraction code is included separately in the bundle for provenance and is not run by these notebooks. Expected totals describe the supplied snapshot; a different snapshot needs new validation and updated findings.

**Scope:** all five boroughs, issuance dates in 2023–2025. Building source: https://data.cityofnewyork.us/dataset/rbx6-tga4 . Street source: https://data.cityofnewyork.us/d/tqtj-sjs8 .


In [ ]:
# Load existing CSVs or ZIP backups locally; upload only when an input is missing.
# Only the requested CSV member is read. ZIP contents are not extracted to disk.
# Raw snapshots are reused; this workflow does not redownload live data.
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
from IPython.display import display
try:
    import google.colab
    BASE = Path('/content')
except ImportError:
    BASE = Path.cwd()
DATA_DIR = BASE / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
def load_saved_csv(filename, **kwargs):
    roots = [DATA_DIR, BASE]
    def locate():
        for folder in roots:
            direct = folder / filename
            if direct.exists():
                return pd.read_csv(direct, **kwargs)
        archives = sorted({p for folder in roots for p in folder.glob('*.zip')})
        for path in archives:
            try:
                with ZipFile(path) as archive:
                    members = [n for n in archive.namelist()
                               if Path(n).name == filename]
                    if len(members) == 1:
                        with archive.open(members[0]) as handle:
                            return pd.read_csv(handle, **kwargs)
            except zipfile.BadZipFile:
                continue
        return None
    import zipfile
    result = locate()
    if result is None:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(f'Place {filename} or its ZIP in {DATA_DIR}') from exc
        print(f'Upload {filename} or the ZIP containing it.')
        files.upload()
        result = locate()
    if result is None:
        raise FileNotFoundError(f'Uploaded files did not contain {filename}.')
    return result

In [ ]:
# Load the two real monthly summaries before attempting integration.
building = load_saved_csv('building_monthly_counts.csv')
street = load_saved_csv('street_monthly_counts.csv')
print('Building:',building.shape,'Street:',street.shape)

In [1]:
# This cell defines the real-data integration function.
# It standardizes and validates both monthly summaries before a one-to-one join.
# An outer join preserves unmatched months for review; missing counts stay unknown.
# It rebuilds one shared date column and verifies that joining preserves totals.
# This cell only defines functions and does not load or download data.

import pandas as pd

JOIN_KEYS = ["borough", "year", "month"]

VALID_BOROUGHS = {
    "BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"
}


def validate_monthly_summary(data, count_column):
    required = JOIN_KEYS + [count_column]

    missing_columns = set(required) - set(data.columns)
    if missing_columns:
        raise ValueError(
            f"Missing columns: {sorted(missing_columns)}"
        )

    # Select only needed fields to avoid redundant columns after joining.
    result = data[required].copy()

    result["borough"] = (
        result["borough"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    if not result["borough"].isin(VALID_BOROUGHS).all():
        raise ValueError("Missing or unrecognized borough labels.")

    for column in ["year", "month", count_column]:
        values = pd.to_numeric(result[column], errors="raise")

        if values.isna().any():
            raise ValueError(f"Missing values in {column}.")

        valid_numbers = (
            values.ge(0)
            & values.lt(float("inf"))
            & values.mod(1).eq(0)
        )

        if not valid_numbers.all():
            raise ValueError(
                f"{column} must contain finite, non-negative integers."
            )

        result[column] = values.astype("int64")

    if not result["year"].between(2023, 2025).all():
        raise ValueError("Years outside the agreed 2023–2025 scope.")

    if not result["month"].between(1, 12).all():
        raise ValueError("Months must be between 1 and 12.")

    if result.duplicated(JOIN_KEYS).any():
        raise ValueError(
            "Multiple rows per borough/year/month. "
            "Supply the overall monthly summary, not category-level data."
        )

    return result


def integrate_permits(building_df, street_df):
    building_data = validate_monthly_summary(
        building_df, "building_permit_count"
    )
    street_data = validate_monthly_summary(
        street_df, "street_permit_count"
    )

    integrated = building_data.merge(
        street_data,
        on=JOIN_KEYS,
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    # Nullable integers preserve unknown counts without converting them to zero.
    for column in ["building_permit_count", "street_permit_count"]:
        integrated[column] = integrated[column].astype("Int64")

    integrated["month_start"] = pd.to_datetime({
        "year": integrated["year"],
        "month": integrated["month"],
        "day": 1
    })

    for source, column in [
        (building_data, "building_permit_count"),
        (street_data, "street_permit_count")
    ]:
        if integrated[column].sum() != source[column].sum():
            raise ValueError(f"Integration changed totals for {column}.")

    integrated = integrated.sort_values(
        JOIN_KEYS
    ).reset_index(drop=True)

    print("Integration coverage:")
    print(integrated["_merge"].value_counts())

    print("\nUnmatched counts remain missing pending coverage review.")
    print("Source totals preserved.")

    return integrated


print("Validated integration function is ready.")

Validated integration function is ready.


In [8]:
# This cell joins the real building and street monthly summaries.
# The validated function performs a one-to-one outer join and preserves totals.
# We then verify all 180 borough-months match, check missing values and save
# both the integrated dataset and its validation evidence.

from pathlib import Path

if "building" not in globals():
    raise RuntimeError(
        "Building data is not loaded. Run the Task 3 building-loading "
        "cell first, then return here."
    )

if "integrate_permits" not in globals():
    raise RuntimeError(
        "Run the updated integration-function cell first."
    )

integrated = integrate_permits(building, street)

assert len(integrated) == 180
assert integrated["_merge"].eq("both").all()
assert not integrated.duplicated(["borough", "year", "month"]).any()

assert integrated["building_permit_count"].sum() == 408_566
assert integrated["street_permit_count"].sum() == 1_951_371

assert integrated[
    ["building_permit_count", "street_permit_count"]
].notna().all().all()

TASK3_OUTPUT_DIR = BASE / "task3_outputs"
TASK3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_summary = pd.DataFrame([
    {"check": "Integrated borough-months", "result": len(integrated)},
    {
        "check": "Matched borough-months",
        "result": int(integrated["_merge"].eq("both").sum())
    },
    {
        "check": "Duplicate join keys",
        "result": int(
            integrated.duplicated(["borough", "year", "month"]).sum()
        )
    },
    {
        "check": "Missing building counts",
        "result": int(integrated["building_permit_count"].isna().sum())
    },
    {
        "check": "Missing street counts",
        "result": int(integrated["street_permit_count"].isna().sum())
    },
    {
        "check": "Building issuance total",
        "result": int(integrated["building_permit_count"].sum())
    },
    {
        "check": "Street permit total",
        "result": int(integrated["street_permit_count"].sum())
    }
])

integrated.to_csv(
    TASK3_OUTPUT_DIR / "integrated_monthly_permits.csv",
    index=False
)
validation_summary.to_csv(
    TASK3_OUTPUT_DIR / "integration_validation.csv",
    index=False
)

print("\nVALIDATION RESULTS")
display(validation_summary)

print("\nINTEGRATED DATA PREVIEW")
display(integrated.head())

print("\nReal integration complete and saved.")

Integration coverage:
_merge
both          180
left_only       0
right_only      0
Name: count, dtype: int64

Unmatched counts remain missing pending coverage review.
Source totals preserved.

VALIDATION RESULTS


,check,result
0,Integrated borough-months,180
1,Matched borough-months,180
2,Duplicate join keys,0
3,Missing building counts,0
4,Missing street counts,0
5,Building issuance total,408566
6,Street permit total,1951371



INTEGRATED DATA PREVIEW


,borough,year,month,building_permit_count,street_permit_count,_merge,month_start
0,BRONX,2023,1,955,5410,both,2023-01-01
1,BRONX,2023,2,989,5584,both,2023-02-01
2,BRONX,2023,3,1341,8823,both,2023-03-01
3,BRONX,2023,4,1076,6649,both,2023-04-01
4,BRONX,2023,5,1099,6541,both,2023-05-01



Real integration complete and saved.


In [2]:
# This cell defines an interactive indexed-trend chart for Task 3's question 5.
# Each series is divided by its borough's mean and multiplied by 100.
# This compares relative monthly patterns despite different permit volumes.
# Only matched months are used, and borough correlations include sample sizes.
# No results are calculated until the real integrated data is supplied.

import plotly.express as px


def make_q5_indexed_trends(data):
    matched = data.loc[
        data["_merge"].eq("both"),
        [
            "borough", "month_start",
            "building_permit_count", "street_permit_count"
        ]
    ].copy()

    if matched.empty:
        raise ValueError("No matched months available.")

    if matched.duplicated(["borough", "month_start"]).any():
        raise ValueError("Duplicate borough-month records.")

    metrics = ["building_permit_count", "street_permit_count"]

    for column in metrics:
        matched[column] = pd.to_numeric(
            matched[column], errors="raise"
        ).astype(float)

    matched["month_start"] = pd.to_datetime(matched["month_start"])
    matched = matched.sort_values(["borough", "month_start"])

    correlation_rows = []

    for borough, group in matched.groupby("borough"):
        usable = group.dropna(subset=metrics)
        varies = all(usable[column].nunique() > 1 for column in metrics)

        r = (
            usable[metrics[0]].corr(usable[metrics[1]])
            if len(usable) >= 3 and varies
            else float("nan")
        )

        correlation_rows.append({
            "borough": borough,
            "matched_months": len(usable),
            "pearson_r": r
        })

    long_data = matched.melt(
        id_vars=["borough", "month_start"],
        value_vars=metrics,
        var_name="series",
        value_name="permit_count"
    )

    means = long_data.groupby(
        ["borough", "series"]
    )["permit_count"].transform("mean")

    # An all-zero series has no meaningful relative-activity index.
    long_data["activity_index"] = (
        long_data["permit_count"]
        .div(means.where(means > 0))
        .mul(100)
    )

    long_data["series"] = long_data["series"].replace({
        "building_permit_count": "Building permits",
        "street_permit_count": "Street permits"
    })

    figure = px.line(
        long_data,
        x="month_start",
        y="activity_index",
        color="series",
        facet_col="borough",
        facet_col_wrap=3,
        markers=True,
        hover_data=["permit_count"],
        title="Q5: Monthly Permit Activity Relative to Each Borough’s Mean",
        labels={
            "month_start": "Month",
            "activity_index": "Activity index (period mean = 100)",
            "series": "Permit series",
            "permit_count": "Original count"
        },
        template="plotly_white",
        height=750
    )

    figure.add_hline(
        y=100, line_dash="dot", line_color="gray"
    )

    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=")[-1]
        )
    )

    correlations = pd.DataFrame(correlation_rows)

    return figure, correlations, long_data


print("Q5 indexed-trend function is ready.")

Q5 indexed-trend function is ready.


In [3]:
# This cell defines Task 3's annual growth comparison chart for question 6.
# Growth is calculated only between consecutive years with 12 matched months.
# A zero previous-year count produces undefined growth, not an infinite value.
# The summary reports the growth-rate gap in percentage points.
# Results are calculated later using the real integrated dataset.

def make_q6_growth_comparison(data):
    metrics = ["building_permit_count", "street_permit_count"]
    keys = ["borough", "year", "month"]

    if data.duplicated(keys).any():
        raise ValueError("Duplicate borough-month records.")

    matched = data.loc[
        data["_merge"].eq("both"),
        keys + metrics
    ].copy()

    if matched.empty:
        raise ValueError("No matched observations available.")

    for column in metrics:
        matched[column] = pd.to_numeric(
            matched[column], errors="raise"
        ).astype(float)

    if matched[metrics].isna().any().any():
        raise ValueError("Matched records contain missing counts.")

    annual = (
        matched.groupby(["borough", "year"], as_index=False)
        .agg(
            months_present=("month", "nunique"),
            building_permit_count=("building_permit_count", "sum"),
            street_permit_count=("street_permit_count", "sum")
        )
    )

    annual = annual.loc[annual["months_present"].eq(12)].copy()

    previous = annual[["borough", "year"] + metrics].copy()
    previous["year"] += 1
    previous = previous.rename(columns={
        column: f"previous_{column}" for column in metrics
    })

    growth = annual.merge(
        previous,
        on=["borough", "year"],
        how="inner",
        validate="one_to_one"
    )

    for column in metrics:
        denominator = growth[f"previous_{column}"]

        growth[f"{column}_growth_pct"] = (
            growth[column]
            .div(denominator.where(denominator > 0))
            .sub(1)
            .mul(100)
        )

    growth["growth_gap_pp"] = (
        growth["building_permit_count_growth_pct"]
        - growth["street_permit_count_growth_pct"]
    )
    growth["absolute_gap_pp"] = growth["growth_gap_pp"].abs()

    long_growth = growth.melt(
        id_vars=["borough", "year"],
        value_vars=[
            "building_permit_count_growth_pct",
            "street_permit_count_growth_pct"
        ],
        var_name="series",
        value_name="growth_percent"
    )

    plot_data = long_growth.dropna(subset=["growth_percent"]).copy()

    if plot_data.empty:
        raise ValueError(
            "No valid consecutive full-year comparisons available."
        )

    plot_data["series"] = plot_data["series"].replace({
        "building_permit_count_growth_pct": "Building permits",
        "street_permit_count_growth_pct": "Street permits"
    })
    plot_data["year"] = plot_data["year"].astype(str)

    figure = px.bar(
        plot_data,
        x="borough",
        y="growth_percent",
        color="series",
        facet_col="year",
        barmode="group",
        title="Q6: Annual Growth in Building and Street Permit Activity",
        labels={
            "borough": "Borough",
            "growth_percent": "Change from previous year (%)",
            "series": "Permit series"
        },
        template="plotly_white"
    )

    figure.update_xaxes(tickangle=-35)

    summary = growth[[
        "borough", "year",
        "building_permit_count_growth_pct",
        "street_permit_count_growth_pct",
        "growth_gap_pp", "absolute_gap_pp"
    ]].sort_values("absolute_gap_pp", ascending=False)

    return figure, summary, annual


print("Q6 annual-growth function is ready.")

Q6 annual-growth function is ready.


In [9]:
# This cell runs Task 3's two research charts using the real integrated dataset.
# Q5 compares relative monthly patterns and reports borough correlations.
# Q6 compares annual growth using complete consecutive years.
# Interactive charts and supporting tables are saved for the website and findings.

required_functions = [
    "make_q5_indexed_trends",
    "make_q6_growth_comparison"
]

for function_name in required_functions:
    if function_name not in globals():
        raise RuntimeError(
            f"Run the cell defining {function_name} first, "
            "then rerun this cell."
        )

fig_q5, q5_correlations, q5_indexed_data = (
    make_q5_indexed_trends(integrated)
)

fig_q5.show()

print("Q5 — CORRELATIONS BY BOROUGH")
display(
    q5_correlations.sort_values(
        "pearson_r", ascending=False
    ).round(4)
)

fig_q6, q6_growth_summary, q6_annual_counts = (
    make_q6_growth_comparison(integrated)
)

fig_q6.show()

print("\nQ6 — ANNUAL GROWTH AND GROWTH-RATE GAPS")
display(q6_growth_summary.round(2))

for figure, filename in [
    (fig_q5, "q5_indexed_monthly_trends.html"),
    (fig_q6, "q6_annual_growth_comparison.html")
]:
    figure.write_html(
        str(TASK3_OUTPUT_DIR / filename),
        include_plotlyjs=True
    )

for table, filename in [
    (q5_correlations, "q5_borough_correlations.csv"),
    (q5_indexed_data, "q5_indexed_monthly_data.csv"),
    (q6_growth_summary, "q6_growth_summary.csv"),
    (q6_annual_counts, "q6_annual_counts.csv")
]:
    table.to_csv(TASK3_OUTPUT_DIR / filename, index=False)

print("\nTask 3 research charts and supporting tables saved.")

Q5 — CORRELATIONS BY BOROUGH


,borough,matched_months,pearson_r
3,QUEENS,36,0.4477
1,BROOKLYN,36,0.4470
2,MANHATTAN,36,0.3467
0,BRONX,36,0.1815
4,STATEN ISLAND,36,-0.0468



Q6 — ANNUAL GROWTH AND GROWTH-RATE GAPS


,borough,year,building_permit_count_growth_pct,street_permit_count_growth_pct,growth_gap_pp,absolute_gap_pp
8,STATEN ISLAND,2024,36.93,-2.88,39.82,39.82
9,STATEN ISLAND,2025,2.89,-15.03,17.92,17.92
0,BRONX,2024,14.59,-0.02,14.62,14.62
6,QUEENS,2024,18.33,5.45,12.89,12.89
2,BROOKLYN,2024,10.48,5.12,5.36,5.36
1,BRONX,2025,2.26,-1.31,3.57,3.57
3,BROOKLYN,2025,0.07,-2.88,2.95,2.95
7,QUEENS,2025,-2.19,-0.08,-2.11,2.11
4,MANHATTAN,2024,2.68,2.30,0.38,0.38
5,MANHATTAN,2025,0.84,0.60,0.24,0.24



Task 3 research charts and supporting tables saved.


### Integration validation and research findings

#### Integration and post-integration cleaning

Both datasets were independently aggregated to borough/year/month,
then joined using a one-to-one outer merge.

All 180 expected borough-month combinations matched. There were no
duplicate join keys or missing permit counts after integration.
No zero imputation was necessary.

The integration preserved 408,566 building issuances and 1,951,371
eligible street permits. Counts were standardized to nullable integer
types, and a single month_start field was reconstructed from the keys,
avoiding redundant date columns.

This integration compares activity within the same borough and month.
It does not match individual building permits to street permits.

#### Question 5: Which boroughs have the most similar monthly patterns?

Each series was divided by its own borough mean and multiplied by 100.
An index of 100 represents average activity during the analysis period.
This allows comparison of relative patterns despite different volumes.

Each borough contributes 36 matched monthly observations.

| Borough | Pearson correlation |
|---|---:|
| Queens | 0.4477 |
| Brooklyn | 0.4470 |
| Manhattan | 0.3467 |
| Bronx | 0.1815 |
| Staten Island | -0.0468 |

Queens and Brooklyn have the strongest observed positive linear
associations, although neither relationship is close to perfect.
Their correlations are nearly identical; the small difference should
not be interpreted as a meaningful ranking.

Manhattan has a smaller positive association, while the Bronx has
a weaker association. Staten Island's correlation is close to zero,
indicating little contemporaneous linear association in this period.

Dividing each series by a positive mean changes its scale, not its
Pearson correlation.

#### Question 6: Where do annual growth rates diverge most?

Growth was calculated for 2024 versus 2023 and 2025 versus 2024,
using complete years with 12 matched months.

Staten Island has the largest divergence. In 2024, building issuance
counts increased by approximately 36.93%, while street permit counts
fell by 2.88%. The gap calculated from unrounded values is 39.82
percentage points.

In 2025, Staten Island building issuances increased by 2.89%, while
street permits fell by 15.03%, producing a 17.92 percentage-point gap.

Manhattan has the smallest absolute growth gaps in both comparisons:
approximately 0.38 percentage points in 2024 and 0.24 in 2025.

The growth gap is building growth minus street growth, expressed in
percentage points. It does not measure unmet infrastructure demand.

#### Limitations and potential use

These are descriptive associations, not evidence that building
activity causes street work or that agencies coordinate their permits.
Shared seasonality, trends, administrative processes and renewal
activity may influence the results.

Monthly observations may be serially related; no statistical
significance or causal conclusions are claimed.

The domains use different operational counting rules and cover
different activities. Permits do not directly measure completed work,
housing production, congestion or infrastructure capacity.

Planners could use divergent patterns to identify boroughs for
further investigation, rather than conclude that street work is
sufficient or insufficient.

### AI assistance and prompt record

ChatGPT assisted with source review, Python implementation, interpretation of diagnostic outputs and notebook organization. Real outputs were reviewed before reporting conclusions. Prompts included:

- “analyse the 3 things with the project description along with the task list” — review of the three contributions and missing requirements.
- “is there anything in the meantime to do in the other 2 tasks?” — preparation of integration and research-chart functions.
- “after every code block can you send me a small comment to add at the top?” — explanatory code comments.

Follow-up messages supplied executed outputs for diagnosis and interpretation. During the final review, ChatGPT reordered cells, replaced repeated live downloads with saved-snapshot loading, removed obsolete code and added validation. The notebook code records the implementation; reported results are grounded in real data rather than generated example values.
